# Sentinel Trade Classifier - Fine-Tuning

Fine-tune Ministral-8B-Instruct on Sentinel trade classification data.

**Requirements:** Free Colab T4 GPU or better.

**Dataset:** `mistral-hackaton-2026/sentinel-trade-classifier` (451 train, 51 test)

**Output:** LoRA adapter pushed to `mistral-hackaton-2026/sentinel-trade-classifier-ministral-8b`

In [ ]:
!pip install -q trl>=0.12.0 peft>=0.7.0 datasets bitsandbytes accelerate transformers

In [ ]:
from huggingface_hub import login
login()  # Enter your HF token with write access

In [ ]:
from datasets import load_dataset

dataset = load_dataset("mistral-hackaton-2026/sentinel-trade-classifier")
train_ds = dataset["train"]
val_ds = dataset["test"]

print(f"Training examples: {len(train_ds)}")
print(f"Validation examples: {len(val_ds)}")
print(f"\nSample:")
print(train_ds[0]["messages"][:2])  # Show system + user messages

In [ ]:
import torch
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# Check GPU
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# LoRA config for efficient fine-tuning
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

# Adjust batch size based on GPU memory
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
if vram_gb < 20:  # T4 (16GB)
    batch_size = 1
    grad_accum = 16
elif vram_gb < 40:  # A10G (24GB)
    batch_size = 2
    grad_accum = 8
else:  # A100 (40/80GB)
    batch_size = 4
    grad_accum = 4

print(f"Using batch_size={batch_size}, grad_accum={grad_accum}")

# SFT training config
training_args = SFTConfig(
    output_dir="sentinel-classifier",
    push_to_hub=True,
    hub_model_id="mistral-hackaton-2026/sentinel-trade-classifier-ministral-8b",
    hub_private_repo=False,

    # Training hyperparameters
    num_train_epochs=3,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_length=512,

    # Eval
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=50,
    hub_strategy="every_save",

    # Logging
    logging_steps=5,

    # Performance
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim="adamw_torch_fused",
    dataloader_num_workers=2,
)

In [ ]:
# Initialize trainer with 4-bit quantization for memory efficiency
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

trainer = SFTTrainer(
    model="mistralai/Ministral-8B-Instruct-2410",
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
    args=training_args,
    model_init_kwargs={"quantization_config": bnb_config},
)

In [ ]:
# Train!
trainer.train()

In [ ]:
# Push final model to Hub
trainer.push_to_hub()
print("Training complete! Model pushed to Hub.")
print("Model: https://huggingface.co/mistral-hackaton-2026/sentinel-trade-classifier-ministral-8b")

In [ ]:
# Quick test: classify a sample
from transformers import pipeline

pipe = pipeline("text-generation", model=trainer.model, tokenizer=trainer.tokenizer, max_new_tokens=256)

test_messages = val_ds[0]["messages"][:2]  # system + user only
print("Input:", test_messages[1]["content"][:200])
print()

result = pipe(test_messages)
print("Prediction:", result[0]["generated_text"][-1]["content"])